# Notebook 02 — Synthetic Images, Brightness Adjustment, and Grid Copying

**Session:** Interpolation and Image upsampling  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Construct synthetic grayscale images (solid black and uniform random noise) using 2D
list comprehensions, copy images with clamped brightness adjustments (`+50` / `-50`),
and observe why naive sparse copying onto a $2\times$ canvas leaves zero/white holes
that motivate interpolation.

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGE_PATH = Path("../data/nature.jpg")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Creating Black and Random-Noise Images

In [ ]:
height, width = 200, 300
black_image = [[0] * width for _ in range(height)]

rng = random.Random(42)
noise_image = [[rng.randint(0, 255) for _ in range(width)] for _ in range(height)]

cv2.imwrite(str(OUTPUT_DIR / "02_black_image.jpg"), np.asarray(black_image, dtype=np.uint8))
cv2.imwrite(str(OUTPUT_DIR / "02_noise_image.jpg"), np.asarray(noise_image, dtype=np.uint8))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(black_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Black Image (0)")
axes[0].axis("off")

axes[1].imshow(noise_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Uniform Noise [0, 255]")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 2. Copying and Brightness Saturation Clamping

In [ ]:
image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE).tolist()
h, w = len(image), len(image[0])

bright_image = [[min(255, image[i][j] + 50) for j in range(w)] for i in range(h)]
dark_image = [[max(0, image[i][j] - 50) for j in range(w)] for i in range(h)]

cv2.imwrite(str(OUTPUT_DIR / "02_nature_bright_50.jpg"), np.asarray(bright_image, dtype=np.uint8))
cv2.imwrite(str(OUTPUT_DIR / "02_nature_dark_50.jpg"), np.asarray(dark_image, dtype=np.uint8))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, img_data, title in zip(
    axes,
    [dark_image, image, bright_image],
    ["Brightness -50", "Original", "Brightness +50"],
    strict=True,
):
    ax.imshow(img_data, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Why Naive 2x Copying Creates Holes

In [ ]:
scale = 2
new_h, new_w = h * scale, w * scale

# Region-copy: leaves bottom-right 3/4 of canvas black
part_canvas = [[0] * new_w for _ in range(new_h)]
for i in range(h):
    for j in range(w):
        part_canvas[i][j] = image[i][j]

# Uniform-stride copy on black (0) and white (255) canvases: leaves gaps between pixels
sparse_black = [[0] * new_w for _ in range(new_h)]
sparse_white = [[255] * new_w for _ in range(new_h)]
for i in range(h):
    for j in range(w):
        sparse_black[i * scale][j * scale] = image[i][j]
        sparse_white[i * scale][j * scale] = image[i][j]

print("Top-left 4x4 of sparse_black:")
for row in sparse_black[:4]:
    print(" ", row[:4])